# LEDGEIT — train the defect model on Colab
Runtime → Change runtime type → **T4 GPU**, then **Run all**. You will be asked to upload `NEU-DET.zip` (the Kaggle dataset, zipped).
Produces two versions: `ledgeit-defect-yolo@1.0.0` (YOLOv8n) and `@1.1.0` (YOLOv8s), plus a `ledgeit_models.zip` to download.

In [ ]:
!nvidia-smi -L
!git clone --branch aryan/chore/ui-polish https://github.com/frchaitanyaaa/LEDGEIT.git
%cd LEDGEIT
!pip install -q -r services/ai-inspector/requirements-yolo.txt

## 1. Upload the dataset (NEU-DET.zip)

In [ ]:
from google.colab import files
import zipfile, pathlib
up = files.upload()
name = next(iter(up))
pathlib.Path('datasets').mkdir(exist_ok=True)
zipfile.ZipFile(name).extractall('datasets/raw')
!find datasets/raw -maxdepth 2 | head

In [ ]:
# the prep script searches recursively, so any zip layout works
!python services/ai-inspector/training/prepare_neu_det.py --src datasets/raw --out datasets/neu-det-yolo --val 0.2
!ls datasets/neu-det-yolo/images/train | wc -l

## 2. Train v1.0.0 (YOLOv8n) and v1.1.0 (YOLOv8s)

In [ ]:
!cd services/ai-inspector && python training/train.py --data ../../datasets/neu-det-yolo/data.yaml --base yolov8n.pt --version 1.0.0 --epochs 100 --device 0 | tee ../../train_1.0.0.log

In [ ]:
!cd services/ai-inspector && python training/train.py --data ../../datasets/neu-det-yolo/data.yaml --base yolov8s.pt --version 1.1.0 --epochs 100 --device 0 | tee ../../train_1.1.0.log

## 3. Update the manifest with the real metrics + dates (no manual JSON editing)

In [ ]:
import json, datetime, pathlib
NL = chr(10)
man_path = pathlib.Path('services/ai-inspector/models/manifest.json')
man = json.loads(man_path.read_text())
for ver in ('1.0.0', '1.1.0'):
    log = pathlib.Path(f'train_{ver}.log').read_text()
    start = log.rindex('{' + NL + '  "file"')
    out = json.loads(log[start:log.rindex('}') + 1])
    for e in man['models']:
        if e['backend'] == 'yolo' and e['version'] == ver:
            e['metrics'] = out['metrics']
            e['training_date'] = datetime.date.today().isoformat()
            print(ver, out['file'], out['weights_sha256'], out['metrics']['mAP50'])
man_path.write_text(json.dumps(man, indent=2) + NL)


## 4. Download results
Copy the two `.pt` files into `services/ai-inspector/models/` and replace `manifest.json` in your repo. **Never commit the .pt files** (git-ignored).

In [ ]:
!cd services/ai-inspector && zip -j ../../ledgeit_models.zip models/*.pt models/manifest.json ../../train_*.log
files.download('ledgeit_models.zip')